In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import importlib

from kfirlib.comparisons import *
from kfirlib.util import *
from kfirlib import gpt

In [ ]:
ROOT = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\HPV\sophie\Leah Borovoi\1556_pdfs\files'
LIST_PATH = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables\fromJamalah9_6\389_for_full_text.xlsx'

In [ ]:
list_df = pd.read_excel(LIST_PATH)
list_df.head()

In [ ]:
invariant = lambda t: extract_letters(t)[:70]
list_df['invariant'] = list_df.Title.apply(lambda t: extract_letters(t)[:70])
titles = set(list_df['invariant'])
len(titles)             

In [ ]:
parsing_regex = re.compile(r'.* - (.*?)\.pdf')

total_files = 0
non_pdfs = 0
debug_rows = []
result_rows = []
for current_dir, subdirs, files in os.walk(ROOT):
    for file in files:
        total_files += 1
        if not file.endswith('.pdf'):
            non_pdfs += 1
            continue
        match = parsing_regex.fullmatch(file)
        row = {
            'filename': file,
            'full_path': os.path.join(current_dir, file),
        }
        if not match:
            print(f'file does not match: {file}')
            debug_rows.append(row)
            continue
        row['title'] = match.group(1)
        extract = invariant(row['title'])
        row['is_in_list'] = extract in titles
        debug_rows.append(row)
        if row['is_in_list']:
            path = os.path.join(current_dir, file)
            result_rows.append({'path': path, 'invariant': extract})            

result_df = pd.DataFrame(result_rows)
total_files, non_pdfs, len(result_df)

In [ ]:
debug_df = pd.DataFrame(debug_rows)
debug_df.head()

In [ ]:
debug_df.to_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables\fromJamalah9_6\matches.xlsx', index=False)

In [ ]:
list_df['found'] = list_df.invariant.isin(result_df.invariant)
list_df['found'].sum()

In [ ]:
list_df.to_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables\fromJamalah9_6\389_with_found.xlsx', index=False)

In [ ]:
list_df['title_extract'] = list_df.title.apply(extract_letters)
len(list_df), list_df['title_extract'].nunique()

In [ ]:
missing_df = list_df[~list_df.found]

In [ ]:
client = gpt._CLIENT

In [ ]:
files_with_paths = []
for path in tqdm(result_df.path):
    files_with_paths.append({'path': path,
        'file_object': client.files.create(
        file=open(path, "rb"),
        purpose="user_data")})


In [ ]:
ROOT2 = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\HPV\sophie\Leah Borovoi\pdfs'
files = os.listdir(ROOT2)
for file in tqdm(files):
    if not file.endswith('.pdf'):
        continue
    path = os.path.join(ROOT2, file)
    files_with_paths.append({'path': path,
        'file_object': client.files.create(
            file=open(path, "rb"),
            purpose="user_data")})

In [ ]:
len(files_with_paths)

In [ ]:
system_prompt = f'''
You are a scientist. You will be given several PDF files of scientific articles. You need to decide whether to include each one of them in a study.

You will be given criteria to decide whether to include or to exclude.

For each article, please print the article's title, then your reasoning, then the decision whether to include or exclude, in the following format:

```
TITLE
<article's title>

REASONING
<reasoning why to include or to exclude>

DECISION
<one of INCLUDE or EXCLUDE>

TITLE
<next article's title>
```


etc. 
You must do this for **each** input.

Write out the decisions for all inputs **in the same order as they are given**.
'''

In [ ]:
def run_model_on_batch(batch):
    content = [
                {
                    "type": "input_text",
                    "text": gpt.JUMALAH_CRITERIA,
                },]
    for item in batch:
        content.append({ "type": "input_file", "file_id": item['file_object'].id,})
    response = client.responses.create(
        model=gpt.Models.MINI,
        input=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": content,
            }
        ]
    )
    return response.output_text

In [ ]:
OUTPUTS = None
TO_PROCESS = None

In [ ]:
def run_on_missing_items(items, batch_size):
    global OUTPUTS, TO_PROCESS
    to_process = [item for item in items if 'title' not in item]
    TO_PROCESS = to_process
    print(f'Running on {len(to_process)} articles')

    outputs = []
    batches = list(batched(to_process, batch_size))
    for batch in tqdm(batches):
        try:
            outputs.append(run_model_on_batch(batch))
        except BaseException as e:
            print(f'Error: {type(e)}, {e}')
            print('Current files:', *[item['path'] for item in batch], sep='\n')

    OUTPUTS = outputs
    missing_items = 0
    updated_items = 0

    for i, (output, batch) in enumerate(zip(outputs, batches)):
        try:
            res = parse_pdf_response(output)
        except Exception as e:
            print(f'Error in batch {i}: {e}')
            missing_items += len(batch)
            continue
        if len(res) != len(batch):
            print(f'Batch {i}: Expected {len(batch)} results but got {len(res)}.')
            missing_items += len(batch)
            continue
        for item, result in zip(batch, res):
            item.update(result)
        updated_items += len(batch)

    print(f'Updated {updated_items} items, missing {missing_items} items.')


In [ ]:
DECISION_DICT = {'INCLUDE': True, 'EXCLUDE': False}

def parse_pdf_response(response: str):
    items = []
    current_type = None

    for line in response.splitlines():
        line = line.strip()
        if not line or line in {'```', '---'}:
            continue
        if line in {'TITLE', 'REASONING', 'DECISION'}:
            current_type = line
            if line == 'TITLE':
                items.append({'title': '', 'reasoning': '', 'decision': ''})
            continue

        if not current_type:
            raise ValueError(f'This line has no key to assign to: "{line}"')
        key = current_type.lower()
        _add_info(items[-1], key, line)

        if current_type == 'DECISION':
            current_type = ''

    for i, item in enumerate(items):
        if not item['title'] or not item['reasoning'] or not item['decision']:
            raise ValueError(f'Bad item {i}: {item}')
        decision = item['decision']
        if decision not in DECISION_DICT:
            raise ValueError(f'Invalid decision in item {i}: {decision}')
        item['decision'] = DECISION_DICT[decision]

    titles = [item['title'] for item in items]
    if len(set(titles)) < len(titles):
        raise ValueError(f'Titles are not unique: {titles}')

    return items

def _add_info(item, key, line):
    current = item[key]
    if current:
        item[key] = '\n'.join([current, line])
    else:
        item[key] = line

In [ ]:
run_on_missing_items(files_with_paths, 5)

In [ ]:
run_on_missing_items(files_with_paths, 3)

In [ ]:
run_on_missing_items(files_with_paths, 1)

doi_10.2340_00015555-3117_4a13b853d0.pdf was not a PDF

In [ ]:
df = pd.DataFrame(files_with_paths)
del df['file_object']
df.head()

In [ ]:
df.to_excel(fr'{ROOT}/../results1.xlsx', index=False)

In [ ]:
df.decision.sum()

In [ ]:
len(df), df.title.nunique()

In [ ]:
df.drop_duplicates('title', inplace=True)

In [ ]:
df['title_extract'] = df.title.apply(extract_letters)
len(df), df.title_extract.nunique()

In [ ]:
df.drop_duplicates('title_extract', inplace=True)

In [ ]:
full_article_df = df

In [ ]:
list_df.columns

In [ ]:
full_article_df.rename(columns={'reasoning': 'full_article_reasoning', 'decision': 'full_article_decision'}, inplace=True)

In [ ]:
combined_df = pd.merge(list_df, full_article_df, on='title_extract', how='outer')
len(combined_df)

In [ ]:
combined_df.columns

In [ ]:
combined_df[['title_x', 'title_y']]

In [ ]:
del combined_df['title_y']
del combined_df['invariant']
del combined_df['found']
combined_df.rename(columns={'title_x': 'title'}, inplace=True)

In [ ]:
path = os.path.join(ROOT, 'full_results.xlsx')
print(path)
combined_df.to_excel(path, index=False)